In [37]:
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-0"

In [54]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,

    }
    
    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

In [48]:
messages = []

add_user_message(messages, "enerate a one sentance movie idea based on the following genre: comedy")
answer = chat(messages, temperature=0) 

answer

'When a perfectionist wedding planner accidentally books the same venue for two completely different weddings on the same day, she must secretly orchestrate both a gothic vampire-themed ceremony and a bright Disney princess celebration while keeping the feuding families from discovering each other.'

Exerise

In [49]:
messages = []

add_user_message(messages, "Write a code that schedules a meeting for next week")
answer = chat(messages, system= "you love clean and concise code") 

answer

'Here\'s a clean and concise meeting scheduler in Python:\n\n```python\nfrom datetime import datetime, timedelta\nfrom dataclasses import dataclass\nfrom typing import List, Optional\n\n@dataclass\nclass Meeting:\n    title: str\n    date: datetime\n    duration_hours: float\n    attendees: List[str]\n    \n    def __str__(self):\n        return f"{self.title} | {self.date.strftime(\'%A, %B %d at %I:%M %p\')} | {self.duration_hours}h"\n\nclass MeetingScheduler:\n    def __init__(self):\n        self.meetings: List[Meeting] = []\n    \n    def schedule_next_week(self, title: str, day: str, time: str, \n                          duration: float = 1.0, attendees: List[str] = None) -> Meeting:\n        """Schedule a meeting for next week"""\n        next_week = datetime.now() + timedelta(weeks=1)\n        meeting_date = self._parse_datetime(next_week, day, time)\n        \n        meeting = Meeting(\n            title=title,\n            date=meeting_date,\n            duration_hours=durat

Streaming

In [50]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01SrLYXpCc7USW7dQnGdfYoV', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='F', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='akeDataGen is a synthetic database containing 50,000 artificially generated customer records with', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta

In [51]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

"UserMetrics3000" is a comprehensive database containing detailed behavioral analytics, purchasing patterns, and demographic information for over 50 million online consumers across North America and Europe.

In [52]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()

JSON Data

In [55]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "Name": "ProcessOrderRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n'

Exercise

In [59]:
messages = []

promt = """
Generate three different AWS CLI commands.
"""
add_user_message(messages, promt)
add_assistant_message(messages, "```bash")

text = chat(messages, stop_sequences=["```"])
text.strip()

'# 1. List all S3 buckets in your AWS account\naws s3 ls\n\n# 2. Describe all EC2 instances in the us-west-2 region\naws ec2 describe-instances --region us-west-2\n\n# 3. Create a new IAM user named "developer"\naws iam create-user --user-name developer'

In [60]:
from IPython.display import Markdown

Markdown(text)


# 1. List all S3 buckets in your AWS account
aws s3 ls

# 2. Describe all EC2 instances in the us-west-2 region
aws ec2 describe-instances --region us-west-2

# 3. Create a new IAM user named "developer"
aws iam create-user --user-name developer
